# 🔍 W06 — Multi-Level Explainability Analysis

**Objective**: Apply three levels of interpretability to the best models.

- **Level 1**: Global Feature Importance (XGBoost tree-based)
- **Level 2**: Temporal Feature Dynamics (Attention weights)
- **Level 3**: Instance-Level SHAP Explanations
- **Synthesis**: Cross-level comparison

**Models**: ML = XGBoost (correlation, 32 feat), DL = BiLSTM+Attn (AFICv, 16 feat)

**Author**: Fatima Khadija Benzine — March 2026

---
## 0. Setup

In [ ]:
import os
if not os.path.exists('/content/PhD-Project-'):
    !git clone https://github.com/f-khadija-benzine/PhD-Project-.git /content/PhD-Project-
!pip install xgboost shap -q
os.chdir('/content/PhD-Project-/src')

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import json, time

project_root = Path('/content/PhD-Project-')
TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M')
SAVE_DIR = f'/content/drive/MyDrive/PhD_results/W06_{TIMESTAMP}'
os.makedirs(SAVE_DIR, exist_ok=True)

from data_loader import MultiDatasetLoader
from preprocessing import DataNormalizer, create_sliding_windows, evaluate_per_unit
from bi_fusion import BIFusionPipeline, CONTINUOUS_BI_VARS
from feature_selection import BIAwareFeatureSelector
from feature_selection_aficv import AFICvFeatureSelector
from ml_branch import MLBranch
from attention import build_dual_attention_bilstm
from explainability import (
    get_tree_importance, plot_tree_importance,
    analyze_attention_temporal, plot_attention_heatmap,
    compute_shap_values, aggregate_shap_to_original,
    plot_shap_summary, plot_shap_waterfall, plot_shap_unit_evolution,
    compare_importance_levels,
)
import tensorflow as tf

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

print(f"GPU: {tf.config.list_physical_devices('GPU')}")
print(f"Save: {SAVE_DIR}")
print("All imports ✓")

---
## 1. Load Data + Best Models

In [ ]:
DATASET = 'FD001'
W = 30
PAD = False

# Load & preprocess (same as W05)
loader = MultiDatasetLoader()
ds = loader.load_cmapss_dataset(DATASET)
meta_cols = ['unit', 'cycle', 'rul']
train_raw = ds['train'].copy()
test_raw = ds['test'].copy()
train_raw['rul'] = train_raw['rul'].clip(upper=125)
if 'rul' in test_raw.columns:
    test_raw['rul'] = test_raw['rul'].clip(upper=125)

sensor_cols = [c for c in train_raw.columns if c.startswith('sensor_')]
setting_cols = [c for c in train_raw.columns if c.startswith('setting_')]

norm = DataNormalizer(method='minmax')
train_norm = norm.fit_transform(train_raw, sensor_cols + setting_cols)
test_norm = norm.transform(test_raw)

fusion = BIFusionPipeline()
train_fused = fusion.fuse(train_norm, DATASET, split='train', encode_categoricals=True)
test_fused = fusion.fuse(test_norm, DATASET, split='test', encode_categoricals=True)
bi_cols = fusion.get_bi_columns(train_fused)
bi_cont = [c for c in CONTINUOUS_BI_VARS if c in train_fused.columns]
bi_norm = DataNormalizer(method='minmax')
train_fused = bi_norm.fit_transform(train_fused, bi_cont)
test_fused = bi_norm.transform(test_fused)

# Correlation features (for ML branch)
sel_corr = BIAwareFeatureSelector(variance_threshold=0.01, correlation_threshold=0.95)
fn_corr = sel_corr.select_features(data=train_fused, sensor_cols=sensor_cols,
    bi_cols=bi_cols, setting_cols=setting_cols, exclude_cols=meta_cols)
tr_corr = sel_corr.transform(train_fused, keep_cols=meta_cols)
te_corr = sel_corr.transform(test_fused, keep_cols=meta_cols)
X_train_corr, y_train_corr = create_sliding_windows(tr_corr, W, fn_corr, 'rul', pad=PAD)
X_test_corr, y_test_corr = create_sliding_windows(te_corr, W, fn_corr, 'rul', pad=PAD)

# AFICv features (for DL branch)
sel_aficv = AFICvFeatureSelector(base_learner='xgboost', n_folds=5, cumulative_threshold=0.90)
fn_aficv = sel_aficv.select_features_stratified(data=train_fused, sensor_cols=sensor_cols,
    bi_cols=bi_cols, setting_cols=setting_cols, target_col='rul', group_col='unit')
tr_aficv = sel_aficv.transform(train_fused, keep_cols=meta_cols)
te_aficv = sel_aficv.transform(test_fused, keep_cols=meta_cols)
X_train_aficv, y_train_aficv = create_sliding_windows(tr_aficv, W, fn_aficv, 'rul', pad=PAD)
X_test_aficv, y_test_aficv = create_sliding_windows(te_aficv, W, fn_aficv, 'rul', pad=PAD)

print(f"Correlation: {len(fn_corr)} features, train={X_train_corr.shape}")
print(f"AFICv:       {len(fn_aficv)} features, train={X_train_aficv.shape}")

In [ ]:
# Load best params from W05
RESULTS_W05 = '/content/drive/MyDrive/PhD_results/W05_20260303_0304'

with open(f'{RESULTS_W05}/A2_ml_grid_best_20260303_0304.json', 'r') as f:
    ml_saved = json.load(f)
with open(f'{RESULTS_W05}/B2_dl_grid_best_20260303_0304.json', 'r') as f:
    dl_saved = json.load(f)

best_ml = ml_saved['params']
best_dl = dl_saved['params']

print(f"ML: {best_ml}")
print(f"DL: {best_dl}")

In [ ]:
# Train ML branch (XGBoost, correlation)
ml_model = MLBranch(model_type='xgboost',
    flatten_strategy=best_ml['flatten_strategy'],
    n_estimators=best_ml['n_estimators'],
    max_depth=best_ml['max_depth'],
    learning_rate=best_ml['learning_rate_xgb'])
ml_model.fit(X_train_corr, y_train_corr, feature_names=fn_corr)
print("ML trained ✓")

# Train DL branch (BiLSTM+Attn, AFICv)
tf.keras.backend.clear_session()
dl_model, attn_model = build_dual_attention_bilstm(
    window_size=W, n_features=len(fn_aficv),
    lstm_units=best_dl['lstm_units'],
    feature_attention_dim=best_dl['feature_attention_dim'],
    temporal_attention_dim=best_dl['temporal_attention_dim'],
    dropout_rate=best_dl['dropout_rate'],
    dense_units=best_dl['dense_units'],
    learning_rate=best_dl['learning_rate'])

dl_model.fit(X_train_aficv, y_train_aficv,
    epochs=100, batch_size=best_dl['batch_size'], validation_split=0.2,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)],
    verbose=1)
print("DL trained ✓")

---
## 2. Level 1 — Global Feature Importance (Tree-Based)

In [ ]:
tree_imp = get_tree_importance(ml_model, fn_corr, top_k=20)
print("=== Global Feature Importance (XGBoost) ===")
print(tree_imp.to_string(index=False))

# Count BI vs sensor in top features
n_bi = (tree_imp['category'] == 'bi').sum()
n_sensor = (tree_imp['category'] == 'sensor').sum()
print(f"\nTop 20: {n_sensor} sensor, {n_bi} BI features")

In [ ]:
plot_tree_importance(tree_imp.head(15),
    save_path=f'{SAVE_DIR}/level1_tree_importance_{TIMESTAMP}.png')

---
## 3. Level 2 — Temporal Feature Dynamics (Attention)

In [ ]:
attn_result = analyze_attention_temporal(attn_model, X_test_aficv, fn_aficv)

print("=== Mean Feature Attention Weights ===")
for i, (name, weight) in enumerate(
    sorted(zip(fn_aficv, attn_result['mean_feature_weights']),
           key=lambda x: -x[1])):
    cat = 'BI' if not name.startswith('sensor') and not name.startswith('setting') else 'Sensor'
    print(f"  {i+1:2d}. {name:25s} {weight:.4f}  [{cat}]")

In [ ]:
plot_attention_heatmap(attn_result,
    save_path=f'{SAVE_DIR}/level2_attention_weights_{TIMESTAMP}.png')

---
## 4. Level 3 — SHAP Explanations

In [ ]:
# Compute SHAP values (takes ~1-2 min)
shap_result = compute_shap_values(ml_model, X_test_corr, fn_corr)
print(f"SHAP values shape: {shap_result['shap_values'].shape}")
print(f"Expected value (baseline): {shap_result['expected_value']:.2f}")

In [ ]:
# SHAP Summary plot
plot_shap_summary(shap_result, max_display=15,
    save_path=f'{SAVE_DIR}/level3_shap_summary_{TIMESTAMP}.png')

In [ ]:
# Aggregate SHAP to original features
shap_agg = aggregate_shap_to_original(shap_result)
print("=== SHAP Importance (aggregated to original features) ===")
print(shap_agg.head(15).to_string(index=False))

n_bi = (shap_agg.head(15)['category'] == 'bi').sum()
print(f"\nTop 15: {15-n_bi} sensor, {n_bi} BI features")

In [ ]:
# Waterfall plot for specific units
# Unit with high RUL (healthy)
plot_shap_waterfall(shap_result, sample_idx=0,
    save_path=f'{SAVE_DIR}/level3_shap_waterfall_healthy_{TIMESTAMP}.png')

In [ ]:
# Waterfall plot for unit near failure (last sample of a unit)
# Find a unit with low true RUL
low_rul_idx = np.argmin(y_test_corr)
print(f"Sample {low_rul_idx}: true RUL = {y_test_corr[low_rul_idx]:.0f}")

plot_shap_waterfall(shap_result, sample_idx=low_rul_idx,
    save_path=f'{SAVE_DIR}/level3_shap_waterfall_critical_{TIMESTAMP}.png')

In [ ]:
# SHAP evolution over time for a specific unit
plot_shap_unit_evolution(shap_result, X_test_corr, te_corr,
    unit_id=1, feature_names=fn_corr, top_k=5,
    window_size=W, pad=PAD,
    save_path=f'{SAVE_DIR}/level3_shap_evolution_unit1_{TIMESTAMP}.png')

---
## 5. Synthesis — Cross-Level Comparison

In [ ]:
# Compare all three levels
# Note: attention uses AFICv features (16), tree/SHAP use correlation (32)
# We compare the features that appear in both sets

comparison = compare_importance_levels(
    tree_imp, shap_agg, attn_result,
    save_path=f'{SAVE_DIR}/synthesis_cross_level_{TIMESTAMP}.png')

print("\n=== Cross-Level Feature Importance ===")
print(comparison.head(15).to_string(index=False))

In [ ]:
# Key finding: which BI features appear in top-10 across levels?
print("\n=== BI Features in Top-10 per Level ===")

top10_tree = set(tree_imp.head(10)['feature'])
top10_shap = set(shap_agg.head(10)['feature'])

bi_in_tree = [f for f in top10_tree if not f.startswith('sensor') and not f.startswith('setting')]
bi_in_shap = [f for f in top10_shap if not f.startswith('sensor') and not f.startswith('setting')]

print(f"Tree importance: {bi_in_tree}")
print(f"SHAP importance: {bi_in_shap}")

if attn_result:
    top10_attn = [fn_aficv[i] for i in np.argsort(attn_result['mean_feature_weights'])[::-1][:10]]
    bi_in_attn = [f for f in top10_attn if not f.startswith('sensor') and not f.startswith('setting')]
    print(f"Attention:       {bi_in_attn}")

In [ ]:
# Save all results
tree_imp.to_csv(f'{SAVE_DIR}/level1_tree_importance_{TIMESTAMP}.csv', index=False)
shap_agg.to_csv(f'{SAVE_DIR}/level3_shap_aggregated_{TIMESTAMP}.csv', index=False)
comparison.to_csv(f'{SAVE_DIR}/synthesis_comparison_{TIMESTAMP}.csv', index=False)

print(f"\n✓ All saved to {SAVE_DIR}")

---
## Files saved:
```
PhD_results/W06_YYYYMMDD_HHMM/
    level1_tree_importance_*.csv
    level1_tree_importance_*.png
    level2_attention_weights_*.png
    level3_shap_summary_*.png
    level3_shap_waterfall_healthy_*.png
    level3_shap_waterfall_critical_*.png
    level3_shap_evolution_unit1_*.png
    level3_shap_aggregated_*.csv
    synthesis_cross_level_*.png
    synthesis_comparison_*.csv
```